# Search NHS entries

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

root_dir = Path.home() / 'scratch/projects/befunddolmetscher'

In [ ]:
import sys

sys.path.append((root_dir / 'search').as_posix())

### load NHS data

In [ ]:
import json

with (root_dir / 'data/nhs_20251024_conditions.json').open('r') as f:
    nhs = json.load(f)
nhs = {d['instance_id']: d for d in nhs}
len(nhs)

In [ ]:
from sqlitevec import SQLiteVec

nhs_table = {}
for lang in ['en']:
    nhs_table[lang] = SQLiteVec(
        table="nhs_doc",
        db_file="nhs.db",
        lang=lang,
    )

### load lookup-tables

Download `*.db` from HeiBox and put it in the same directory.

In [ ]:
from helpers import load_dictionary_tables, load_data_tables

icd10_table, mesh_table, do_table, pschyrembel_table = load_dictionary_tables()

In [ ]:
from sqlitevec import SQLiteVec

msd_table = {}
for lang in ['en', 'de']:
    msd_table[lang] = SQLiteVec(
        table=f"msd_doc_{lang}",
        db_file="msd.db",
        lang=lang,
    )

In [ ]:
# placeholder to store the ids
external_ids = {}
for instance_id, r in nhs.items():
    tmp = {k: set([]) for k in ['ICD-10CM', 'MeSH', 'DOID', 'Pschyrembel', 'MSD']}
    for k, v in r['external_ids'].items():
        if isinstance(v, list):
            for u in v:
                if isinstance(u, dict):
                    tmp[k].add(u['value'])
    external_ids[instance_id] = tmp

In [ ]:
external_ids['ff462668b0e25343373d30862e4d7c44']

In [ ]:
with (root_dir / 'data/nhs_alternative_names.json').open('r') as f:
    alternative_names = json.load(f)

### lookup NHS entries in the dictionaries

In [ ]:
from helpers import lookup_title, _add_to_dict
from tqdm.auto import tqdm

for instance_id, r in tqdm(nhs.items()):
    queries = set([])
    lang = 'en'
    d = r['en']['amateur']
    queries.add(d['topic']['name'])
    for name in d['topic']['alternative_name'] + alternative_names.get(instance_id, []):
        queries.add(name)

    for keys, dict_table in [
        (['icd10'], icd10_table),
        (['descriptor_ui'], mesh_table),
        (['doid', 'icd10', 'mesh'], do_table),
        (['pschyrembel_id', 'icd10'], pschyrembel_table),
        (['instance_id'], msd_table),
    ]:
        for query in queries:
            hits = lookup_title(table=dict_table[lang], query=query, keys=keys)
            external_ids = _add_to_dict(hits, external_ids, instance_id)

In [ ]:
external_ids['ff462668b0e25343373d30862e4d7c44'] # Parkinson disease

### assign found external_ids with distance

In [ ]:
from helpers import search_by_similarity, check_by_similarity
from tqdm.auto import tqdm

for instance_id, r in tqdm(nhs.items()):
    emb = {}
    for lang in ['en']:
        found = nhs_table[lang].metadata_search(instance_id=instance_id, audience="amateur")
        emb[lang] = found[0]['embedding']

    for name, key, dict_table in [
        ('ICD-10CM', 'icd10', icd10_table),
        ('MeSH', 'descriptor_ui', mesh_table),
        ('DOID', 'doid', do_table),
        ('Pschyrembel', 'pschyrembel_id', pschyrembel_table),
        ('MSD', 'instance_id', msd_table),
    ]:
        if name not in r['external_ids'] or r['external_ids'][name] is None:
            if len(external_ids[instance_id][name]) > 0:
                ids = list(external_ids[instance_id][name])
                distances = check_by_similarity(dict_table, emb, ids, key, limit=25)
            else:
                distances = search_by_similarity(nhs_table, dict_table, emb, instance_id, key, limit=5)
            r['external_ids'][name] = [{'value': k, 'distance': v} for k, v in distances.items()] if len(distances) > 0 else None


In [ ]:
nhs['ff462668b0e25343373d30862e4d7c44']['external_ids']

In [ ]:
with (root_dir / 'data/nhs_20251029.json').open('w') as f:
    json.dump([r for _, r in nhs.items()], f, indent=4)

### add texts to the database

In [ ]:
import json
output_path = (root_dir / 'results/nhs_embed.jsonl')

vectors = [json.loads(l) for l in output_path.read_text().splitlines()]
vectors = {(v['instance_id'], v['language'], v['edition'], v['paragraph_id']): v['embedding'][0] for v in vectors if len(v['embedding'][0]) == 4096}
len(vectors)

In [ ]:
from tqdm.auto import tqdm

titles = []
texts = []
metadatas = []
embeds = []

for instance_id, r in tqdm(nhs.items()):
    d = r['en']['amateur']
    titles.append(d['topic']['name'])
    texts.append(d['plain_text'])
    metadatas.append({'instance_id': instance_id, 'language': 'en', 'audience': 'amateur'})
    embeds.append(vectors[(instance_id, 'en', 'amateur', d['paragraphs'][0]['paragraph_id'])])

In [ ]:
from sqlitevec import SQLiteVec

nhs_table = {}
for lang in ['en']:
    nhs_table[lang] = SQLiteVec.from_texts(
        titles=titles,
        texts=texts,
        embeds=embeds,
        metadatas=metadatas,
        table="nhs_doc",
        db_file="nhs.db",
        lang=lang,
    )